# Python Dersi 16: Mini Proje, Yapılacaklar Listesi Uygulaması

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/python-dersleri/blob/main/16-yapilacaklar-listesi-projesi.ipynb)

Bu not defteri, haticeozbolat.com'daki **Python Dersi 16** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

Bu derste yeni bir konu yok. Şimdiye kadar öğrendiklerimizi birleştirip kapatıp açınca da görevlerini hatırlayan bir yapılacaklar listesi yazıyoruz:

| Parça | Ders |
|---|---|
| Listeler ve sözlükler | 7-8 |
| Fonksiyonlar | 9 |
| while döngüsü ve menü | 6 ve 10 |
| try ve except | 13 |
| Dosyalar ve CSV | 14 |
| Modüller (`csv`, `datetime`) | 15 |

## 1. Plan

Her görev bir sözlük: `{"no": 1, "gorev": "Ders 16'yı oku", "tarih": "2026-11-17", "bitti": False}`. Tüm görevler bir liste. Liste bir CSV dosyasında saklanıyor.

Programı küçük fonksiyonlara bölüyoruz; her biri tek bir iş yapıyor:

- `yukle()` ve `kaydet()`: dosyayla konuşan iki fonksiyon
- `ekle()`, `tamamla()`, `sil()`, `listele()`: kullanıcının yaptığı işler
- `menu()`: hepsini bir araya getiren döngü

## 2. Fonksiyonlar

In [ ]:
import csv
from datetime import date

DOSYA = "gorevler.csv"
ALANLAR = ["no", "gorev", "tarih", "bitti"]


def yukle():
    """Görevleri dosyadan okur. Dosya yoksa boş liste döner."""
    try:
        with open(DOSYA, "r", encoding="utf-8", newline="") as f:
            gorevler = list(csv.DictReader(f))
    except FileNotFoundError:
        return []
    for g in gorevler:
        g["no"] = int(g["no"])
        g["bitti"] = g["bitti"] == "True"
    return gorevler


def kaydet(gorevler):
    with open(DOSYA, "w", encoding="utf-8", newline="") as f:
        yazici = csv.DictWriter(f, fieldnames=ALANLAR)
        yazici.writeheader()
        yazici.writerows(gorevler)


def ekle(gorev):
    gorevler = yukle()
    yeni_no = max([g["no"] for g in gorevler], default=0) + 1
    gorevler.append({"no": yeni_no, "gorev": gorev, "tarih": date.today().isoformat(), "bitti": False})
    kaydet(gorevler)
    print(f"Eklendi: {yeni_no}. {gorev}")


def bul(gorevler, no):
    for g in gorevler:
        if g["no"] == no:
            return g
    return None


def tamamla(no):
    gorevler = yukle()
    g = bul(gorevler, no)
    if g is None:
        print(f"{no} numaralı görev yok.")
        return
    g["bitti"] = True
    kaydet(gorevler)
    print(f"Tamamlandı: {g['gorev']}")


def sil(no):
    gorevler = yukle()
    g = bul(gorevler, no)
    if g is None:
        print(f"{no} numaralı görev yok.")
        return
    gorevler.remove(g)
    kaydet(gorevler)
    print(f"Silindi: {g['gorev']}")


def listele():
    gorevler = yukle()
    if not gorevler:
        print("Listen boş. Bir görev ekleyerek başla.")
        return
    for g in gorevler:
        isaret = "[x]" if g["bitti"] else "[ ]"
        print(f"{isaret} {g['no']}. {g['gorev']}  ({g['tarih']})")
    biten = sum(g["bitti"] for g in gorevler)
    print(f"\n{len(gorevler)} görevden {biten} tanesi bitti.")

Dikkat edilecek iki yer:

- CSV'den okunan her şey metin olarak gelir (Ders 14). Bu yüzden `yukle()` içinde `no`'yu `int()` ile sayıya, `bitti`'yi `== "True"` ile mantıksal değere çeviriyoruz.
- `max([...], default=0)`: Liste boşsa `max` hata verir. `default=0` ile boş listede 0 dönüyor, ilk görevin numarası 1 oluyor.

## 3. Deneyelim

Önce temiz bir başlangıç için eski dosyayı silelim (varsa).

In [ ]:
import os
if os.path.exists(DOSYA):
    os.remove(DOSYA)

listele()
ekle("Python Dersi 16'yı oku")
ekle("Alıştırmaları çöz")
ekle("Kodu GitHub'a yükle")
print()
listele()

In [ ]:
tamamla(1)
sil(3)
tamamla(7)
print()
listele()

Şimdi dosyanın içine bakalım. Görevler gerçekten kaydedilmiş mi?

In [ ]:
with open(DOSYA, encoding="utf-8") as f:
    print(f.read())

## 4. Menü

Son parça, kullanıcıdan seçim alan bir `while` döngüsü. Sayı beklediğimiz yerlerde `try` ve `except` ile harf girilmesine karşı önlem alıyoruz.

In [ ]:
def sayi_iste(mesaj):
    try:
        return int(input(mesaj))
    except ValueError:
        print("Lütfen bir sayı gir.")
        return None


def menu():
    while True:
        print("\n1) Listele  2) Ekle  3) Tamamla  4) Sil  5) Çıkış")
        secim = input("Seçimin: ").strip()
        if secim == "1":
            listele()
        elif secim == "2":
            gorev = input("Görev: ").strip()
            if gorev:
                ekle(gorev)
        elif secim == "3":
            no = sayi_iste("Görev numarası: ")
            if no is not None:
                tamamla(no)
        elif secim == "4":
            no = sayi_iste("Görev numarası: ")
            if no is not None:
                sil(no)
        elif secim == "5":
            print("Görüşmek üzere!")
            break
        else:
            print("1 ile 5 arasında bir seçim yap.")

Aşağıdaki hücreyi çalıştır ve menüyü kullan. Çıkmak için 5 yaz.

Sonra hücreyi bir daha çalıştırıp 1'i seç: Görevlerin hâlâ orada, çünkü dosyada duruyorlar.

In [ ]:
menu()

## Kendin geliştir

1. **Önceliğe göre:** Her göreve `oncelik` alanı ekle (yüksek, orta, düşük) ve `listele()` önce yüksek öncelikli görevleri göstersin. İpucu: `ALANLAR` listesine de eklemeyi unutma.
2. **Bitenleri temizle:** Menüye "6) Bitenleri sil" seçeneği ekle.
3. **Son tarih:** Görev eklerken bir son tarih iste ve süresi geçmiş görevlerin yanına "(gecikti)" yaz. İpucu: `date.fromisoformat()`.
4. **Arama:** İçinde belirli bir kelime geçen görevleri listeleyen bir `ara(kelime)` fonksiyonu yaz.

## Sıradaki ders

**Python Dersi 17: Sınıflar ve Nesneler, OOP'ye Giriş** [YAZI LİNKİ]

Bu projedeki her görev bir sözlüktü. Bir sonraki derste görevi kendi metotları olan bir nesneye çevirmenin yolunu öğreneceğiz.